*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*
> This notebook contains the raw code for Chapter 9: From Native PyTorch to LightningModule. It shows how the native loop is gradually reorganized into reusable Lightning hooks and Trainer-managed execution.

The core idea of Lightning is simple: keep the model definition focused on its math, while the Trainer handles loops, device placement, logging, and optimization orchestration.

## Anatomy of a LightningModule: 
### Define the Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl

# Lightning keeps the model logic in hooks while the Trainer owns execution details.
class LitClassifier(pl.LightningModule):
    # Step 1: Define the __init__()
    def __init__(
        self,
        in_features: int = 784,
        num_classes: int = 10,
        lr: float = 1e-3,
    ):
        super().__init__()
        # Store constructor parameters in checkpoints and logs.
        self.save_hyperparameters()

        self.backbone = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    # Step 2: Define the forward() method
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)
    
    # Step 3: Define the training_step() method
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        self.log(
            "train_loss",
            loss,
            on_step=True,
            on_epoch=True,
            prog_bar=True,
            batch_size=x.size(0),
        )
        return loss
    
    # Step 4: Add validation, testing and prediction
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        val_loss = F.cross_entropy(logits, y)
        accuracy = (logits.argmax(dim=1) == y).float().mean()

        self.log_dict(
            {"val_loss": val_loss, "val_acc": accuracy},
            on_epoch=True,
            prog_bar=True,
            sync_dist=True,
            batch_size=x.size(0),
        )

        return val_loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        test_loss = F.cross_entropy(logits, y)
        self.log(
            "test_loss", test_loss,
            on_epoch=True, sync_dist=True,
            batch_size=x.size(0),
        )

    def predict_step(self, batch, batch_idx):
        x, _ = batch
        return self(x).argmax(dim=1)
    
    # Step 5: Configure Optimization Separately
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=10
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
            },
        }

### Verify Predictions and Loss Shape

In [ ]:
smoke_model = LitClassifier(
    in_features=784,
    num_classes=10,
    lr=1e-3,
)
smoke_inputs = torch.randn(4, 784)
smoke_targets = torch.randint(0, 10, (4,))

# The LightningModule still behaves like a normal nn.Module before training starts.
smoke_logits = smoke_model(smoke_inputs)
smoke_loss = F.cross_entropy(smoke_logits, smoke_targets)

assert smoke_logits.shape == (4, 10)
assert smoke_loss.ndim == 0
assert "lr" in smoke_model.hparams

## The Modern Execution Engine: L.Trainer & Granular Compilation

### Step 1: Prepare a Learnable Dataset

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, TensorDataset

# A small synthetic dataset gives a learnable target and avoids unrelated randomness.
generator = torch.Generator().manual_seed(42)
teacher = torch.randn(784, 10, generator=generator)

def make_dataset(size):
    features = torch.randn(
        size, 784, generator=generator
    )
    labels = (features @ teacher).argmax(dim=1)
    return TensorDataset(features, labels)

train_data = make_dataset(1000)
val_data = make_dataset(200)

num_workers = min(4, os.cpu_count() or 1)
use_cuda = torch.cuda.is_available()

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=use_cuda,
    persistent_workers=num_workers > 0,
)
val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=use_cuda,
    persistent_workers=num_workers > 0,
)

batch_inputs, batch_targets = next(iter(train_loader))
assert batch_inputs.shape == (32, 784)
assert batch_targets.shape == (32,)

### Step 2: Compile the Tensor-Heavy Region

In [ ]:
model = LitClassifier(
    in_features=784,
    num_classes=10,
    lr=1e-3,
)

# Enable only after benchmarking this model and workload.
# Make sure to use the latest PyTorch version and check the release notes for any updates on torch.compile.
use_compile = False
if use_compile:
    model.backbone = torch.compile(model.backbone)

### Step 3: Configure and Launch the Trainer

In [10]:
precision = "16-mixed" if use_cuda else "32-true"

trainer = pl.Trainer(
    accelerator="auto",
    devices="auto",
    precision=precision,
    max_epochs=10,
    gradient_clip_val=1.0,
    log_every_n_steps=10
)

trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type       | Params | Mode 
------------------------------------------------
0 | backbone | Sequential | 203 K  | train
------------------------------------------------
203 K     Trainable params
0         Non-trainable params
203 K     Total params
0.814     Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
